## **Telecom Customer Support**

### **Stage 1: LangChain (create_agent)**

##### **Importing Libraries**

In [1]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

##### **Model**

In [2]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

api_key = os.getenv("telecom_assg2")

if not api_key:
    raise RuntimeError("Set telecom_assg2 in your .env file before running the agent cells.")

model_name = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

model = ChatGroq(model= model_name, temperature= 0, api_key= api_key)

##### **Mock Telecom Database**

In [3]:
customer_details = {
    "AC10234": {"name": "L. Kim",       "phone_number": "212-555-0142", "plan": "unlimited_plus", "area_code": "212", "monthly_bill_usd": 85,  "data_used_gb": 42},
    "AC20458": {"name": "D. Osei",      "phone_number": "310-555-0198", "plan": "standard_20gb",  "area_code": "310", "monthly_bill_usd": 45,  "data_used_gb": 12},
    "AC30671": {"name": "P. Novak",     "phone_number": "404-555-0113", "plan": "basic_5gb",      "area_code": "404", "monthly_bill_usd": 30,  "data_used_gb": 4.5},
    "AC40892": {"name": "R. Alvarez",   "phone_number": "512-555-0176", "plan": "business_50gb",  "area_code": "512", "monthly_bill_usd": 120, "data_used_gb": 38},
    "AC50103": {"name": "S. Chen",      "phone_number": "606-555-0159", "plan": "standard_20gb",  "area_code": "606", "monthly_bill_usd": 45,  "data_used_gb": 15},
    "AC60217": {"name": "T. Barros",    "phone_number": "213-555-0134", "plan": "basic_5gb",      "area_code": "213", "monthly_bill_usd": 30,  "data_used_gb": 6.2},
    "AC70345": {"name": "N. Whitfield", "phone_number": "718-555-0187", "plan": "unlimited_plus", "area_code": "718", "monthly_bill_usd": 85,  "data_used_gb": 55},
    "AC80456": {"name": "M. Kowalski",  "phone_number": "702-555-0121", "plan": "business_50gb",  "area_code": "702", "monthly_bill_usd": 120, "data_used_gb": 21}
}

plan_details = {
    "basic_5gb":      {"data_gb": 5,           "minutes": "unlimited", "price_usd": 30,  "overage_fee_per_gb": 10},
    "standard_20gb":  {"data_gb": 20,          "minutes": "unlimited", "price_usd": 45,  "overage_fee_per_gb": 8},
    "unlimited_plus": {"data_gb": "unlimited", "minutes": "unlimited", "price_usd": 85,  "overage_fee_per_gb": 0},
    "business_50gb":  {"data_gb": 50,          "minutes": "unlimited", "price_usd": 120, "overage_fee_per_gb": 6},
    "family_100gb":   {"data_gb": 100,         "minutes": "unlimited", "price_usd": 150, "overage_fee_per_gb": 5}
}

network_status = {
    "212": {"status": "Outage",      "outage_hours": 18, "affected_services": ["voice", "data"],       "region": "New York, NY",    "technician_dispatched": True},
    "310": {"status": "Operational", "outage_hours": 0,  "affected_services": [],                      "region": "Los Angeles, CA", "technician_dispatched": False},
    "404": {"status": "Degraded",    "outage_hours": 3,  "affected_services": ["data"],                "region": "Atlanta, GA",     "technician_dispatched": False},
    "512": {"status": "Operational", "outage_hours": 0,  "affected_services": [],                      "region": "Austin, TX",      "technician_dispatched": False},
    "606": {"status": "Outage",      "outage_hours": 30, "affected_services": ["voice", "data", "sms"],"region": "Lexington, KY",   "technician_dispatched": True},
    "213": {"status": "Operational", "outage_hours": 0,  "affected_services": [],                      "region": "Los Angeles, CA", "technician_dispatched": False},
    "718": {"status": "Degraded",    "outage_hours": 5,  "affected_services": ["voice"],               "region": "Brooklyn, NY",    "technician_dispatched": True},
    "702": {"status": "Operational", "outage_hours": 0,  "affected_services": [],                      "region": "Las Vegas, NV",   "technician_dispatched": False}
}

##### **Defining Tools**

In [4]:
# TOOL 1
def lookup_account(customer_id: str):
    """Look up an account by its account_id, e.g., 'AC10234'"""
    
    customer_info = customer_details.get(customer_id.upper())
    
    # If customer information not found, return an error message
    if not customer_info:
        return {"error": f"No information found for {customer_id}."}    
    return customer_info
    
# TOOL 2
def check_network_status(area_code: str | int):
    """Check the current network status for a 3-digit area code, e.g. '212'."""
    
    area_code = str(area_code).strip()

    # Look up the network status.
    status = network_status.get(area_code)
    
    # If area code is not recognized, return an error message
    if not status:
        return {"error": f"No network status found for area code '{area_code}"}
    return status    

# TOOL 3
def request_plan_change(customer_id: str, new_plan: str):
    """Check whether a requested plan exists and calculate the monthly price difference. 
    Does NOT change the plan — only quotes it."""

    account = lookup_account(customer_id)
    
    # Check whether the customer exists
    if "error" in account:
        return {"error": f"Unknown customer ID '{customer_id}'"}
    
    new_plan = new_plan.lower()
    target = plan_details.get(new_plan)
    
    # Check whether the requested plan exists, If target is None, the requested plan does not exist.
    if not target:
        return {"available": False, "reason": f"Unknown plan '{new_plan}'. Valid: basic_5gb, standard_20gb, unlimited_plus, business_50gb."}
    
    current = plan_details[account["plan"]]
    
    # Calculate the price difference
    price_diff = round(target["price_usd"] - current["price_usd"], 2)
    
    # Return the plan-change quotation
    return {"available": True, "current_plan": account["plan"], "new_plan": new_plan, "price_diff_usd": price_diff}
  
tools = [lookup_account, check_network_status, request_plan_change]

support_system_prompt = """You are TeleAssist, a telecom customer support agent. 
Instructions:
- A customer ID such as AC50103 must NEVER be used as an area code.
- When a customer ID is provided, call lookup_account first.
- Use the area_code returned by lookup_account with check_network_status.
- For a plan change, call request_plan_change using the customer ID and requested plan.
- If the user asks multiple questions, complete ALL required tool calls before answering.
- If the user asks multiple questions, call all necessary tools.
- Always end with a structured response.
"""

##### **Agent**

In [5]:
class AgentResponse(BaseModel):
    response: str = Field(description= "The agent's response to the user query.")
    category: Literal["network_status", "account", "plan_change", "other"] = Field(description= "What the query was about.")
    summary: str = Field(description= "A short summary of what was looked up. Empty string if nothing was looked up.")
        
stage1_agent = create_agent(
    model= model,
    tools= tools,
    system_prompt= support_system_prompt,
    response_format= AgentResponse,
    middleware= [PIIMiddleware("email", strategy= "redact"),
                 PIIMiddleware("credit_card", strategy= "block")]
)        

result = stage1_agent.invoke({
    "messages": [{"role": "user",
                  "content": "My customer id is AC70345 — Is my area having a network outage and what would it cost to change my plan to business_50gb??"}]
})

structured = result["structured_response"]
print(structured)

response='Your current plan is unlimited_plus with a monthly bill of $85. Changing to business_50gb will cost $35 more per month. Additionally, there is a network outage in your area with 5 hours of voice service degradation.' category='plan_change' summary='lookup_account for AC70345 and request_plan_change to business_50gb and check_network_status for area code 718'
